# Graph Data Science on RDF — the Neo4j GDS playbook on a `.rete` file

**Everything in this notebook runs in your browser tab** — the Python is
[Pyodide](https://pyodide.org), the UI is JupyterLite, and the graph is a single
1 MB **`.rete`** file on object storage, read lazily over HTTP range requests.
No database server, no install.

[Neo4j Graph Data Science](https://neo4j.com/docs/graph-data-science/current/algorithms/)
works in three steps: **project** a graph out of the store into memory, **run
algorithms** on the projection, **write back** the results. That architecture is not
Neo4j-specific — here we do exactly the same over RDF:

1. **Project** — a SPARQL query pulls an edge list out of the `.rete` file
   (only the byte ranges it touches are downloaded);
2. **Compute** — NetworkX, NumPy and scikit-learn run the GDS algorithm catalog
   in the browser;
3. **Write back** — the results become new triples in a *derived* `.rete` file.

**The dataset**: [Mark Lombardi Networks](https://lombardinetworks.net) — 51
*Narrative Structures*, the network drawings in which Lombardi (1951–2000) mapped
global-finance scandals (BCCI, Iran–Contra, Harken Energy, the Vatican bank…) by
hand: **2,934 actors** and **4,205 arcs whose line style carries the relationship
type**. He computed centrality by eye; we get to check his work.

Run the cells top to bottom (`Shift+Enter`). Dataset license: CC BY-NC-SA 4.0.

## The GDS algorithm catalog, translated

| Neo4j GDS | Here | Runs |
|---|---|---|
| `gds.graph.project` / Cypher projection | SPARQL SELECT → DataFrame → NetworkX / CSR | browser |
| Filtered / subgraph projection | `FILTER` / `VALUES` in the projection query | browser |
| *(no equivalent)* — **projection by ontology** | `reason=True`: OWL `subPropertyOf` hierarchy widens the edge set | browser |
| Degree Centrality | pure SPARQL `GROUP BY` — no export needed | browser |
| PageRank, Article Rank | `nx.pagerank` | browser |
| Betweenness (sampled) | `nx.betweenness_centrality(k=…)` | browser |
| Closeness / Harmonic / Eigenvector | `nx.closeness_centrality`, … | browser |
| HITS | `nx.hits` | browser |
| Articulation Points, Bridges | `nx.articulation_points`, `nx.bridges` | browser |
| Louvain, Modularity | `nx.community.louvain_communities` | browser |
| Leiden | `sknetwork`/`leidenalg` | native |
| Label Propagation | `nx.community.label_propagation_communities` | browser |
| WCC / SCC | `nx.connected_components`, `nx.strongly_connected_components` | browser |
| Triangle Count / Clustering Coefficient | `nx.triangles`, `nx.average_clustering` | browser |
| K-Core Decomposition | `nx.core_number` | browser |
| K-1 Coloring | `nx.greedy_color` | browser |
| K-Means / HDBSCAN (on embeddings) | `sklearn.cluster` | browser |
| Conductance | `nx.conductance` | browser |
| Node Similarity (Jaccard/Overlap/Cosine) | set ops / `scipy.spatial.distance` | browser |
| K-Nearest Neighbors | `sklearn.neighbors.NearestNeighbors` | browser |
| Dijkstra / A* / Yen / Bellman-Ford | `nx.shortest_path`, `nx.astar_path`, `nx.shortest_simple_paths` | browser |
| BFS / DFS / All-pairs shortest path | `nx.bfs_tree`, `nx.all_pairs_shortest_path_length` | browser |
| Spanning trees / Steiner / Max-flow | `nx.minimum_spanning_tree`, `nx.approximation.steiner_tree`, `nx.maximum_flow` | browser |
| Topological sort / DAG longest path | `nx.topological_sort`, `nx.dag_longest_path` | browser |
| Random Walk | a few lines of NumPy | browser |
| **FastRP** node embeddings | ~15 lines of NumPy (below) | browser |
| Node2Vec | random walks + `gensim` | native |
| GraphSAGE / HashGNN | PyTorch Geometric | native |
| Adamic-Adar / Common Neighbors / Pref. Attachment / Resource Allocation | `nx.adamic_adar_index`, … | browser |
| Node classification / regression pipelines | `sklearn` pipelines (below) | browser |
| Link prediction pipelines | `sklearn` on pair features | browser |
| Training methods (LogReg / RF / MLP) | `sklearn.linear_model`, `.ensemble`, `.neural_network` | browser |
| `gds.graph.writeNodeProperties` (write-back) | bake results into a **derived `.rete`** (below) | browser |
| Pregel API | Python loop over the CSR / native frameworks | either |

**Scale honesty:** in-browser NetworkX is comfortable to ~10⁵–10⁶ edges. Past
that, run the *same code* in native Python — or hand the same CSR matrix to
[scikit-network](https://scikit-network.readthedocs.io) and the same edge list to
[PyTorch Geometric](https://pytorch-geometric.readthedocs.io) (both demoed at the
end). Because `.rete` reads are lazy, you only ever download what you project.

In [ ]:
%pip install rete-graph pandas networkx scipy scikit-learn matplotlib

import sys, numpy as np, pandas as pd, networkx as nx
import rete_graph as rete
print(f"rete-graph {rete.__version__} · networkx {nx.__version__} · platform {sys.platform!r}")

In [ ]:
g = rete.open("https://data.graphplaza.com/lombardi/lombardi.rete")
s = g.stats()
print(f"{g.quads:,} triples reachable; opened after fetching only "
      f"{s['bytes']:,} of {s['fileLength']:,} bytes in {s['requests']} range requests")
print(g.card()["title"], "—", g.card()["license"])

PREFIXES = """
PREFIX lomb: <https://w3id.org/rete/lombardi/>
PREFIX lo:   <http://www.lombardinetworks.net/lombardi.owl#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX schema: <https://schema.org/>
"""

## 1 · Graph projection — `gds.graph.project` in SPARQL

GDS projects node and relationship tables out of the store. In this graph every
drawn arc is *reified*: an `Arc` resource with `lomb:source`, `lomb:target`,
`lomb:arcType` (the line style — influence, money, blocked deal…) and
`lomb:inDrawing`. One SPARQL query is the whole projection — **including the
filters**, which in GDS would be relationship-type and node-label projection
config: we drop chronology arrows (`YearArrow`, …) and Lombardi's year-mark
pseudo-actors (`lo:Year`), keeping only substantive relationships:

In [ ]:
edges = g.query_df(PREFIXES + """
SELECT ?src ?tgt ?type ?drawing WHERE {
  ?arc lomb:source ?src ; lomb:target ?tgt ;
       lomb:arcType ?type ; lomb:inDrawing ?drawing .
  FILTER(?type NOT IN (lo:YearArrow, lo:YearLine, lo:Final, lo:SingleNearby))
  FILTER(?src != ?tgt)
  FILTER NOT EXISTS { ?src a lo:Year }   FILTER NOT EXISTS { ?tgt a lo:Year }
  FILTER NOT EXISTS { ?src a lo:YearFinal } FILTER NOT EXISTS { ?tgt a lo:YearFinal }
}""")

names = g.query_df(PREFIXES + """
SELECT ?actor ?label WHERE { ?actor skos:prefLabel ?label ; lomb:actorId ?id }""")
label = dict(zip(names["actor"], names["label"]))

print(f"projected {len(edges):,} typed edges over {len(names):,} labelled actors")
edges["type"].map(lambda t: t.split("#")[-1]).value_counts().to_frame("arcs")

### Projection by ontology — something GDS cannot do

Every arc predicate (`lomb:rel/financialTransaction`, `lomb:rel/influenceControl`, …)
is declared `rdfs:subPropertyOf lomb:connectedTo` in the file's own OWL ontology.
Ask for the *super*-property with OWL 2 QL reasoning on, and the ontology **is**
the projection config — no re-projection, no duplication:

In [ ]:
plain    = len(g.query(PREFIXES + "SELECT ?s ?o WHERE { ?s lomb:connectedTo ?o }"))
entailed = len(g.query(PREFIXES + "SELECT ?s ?o WHERE { ?s lomb:connectedTo ?o }", reason=True))
print(f"asserted connectedTo edges: {plain:,}")
print(f"with subPropertyOf entailment: {entailed:,}  ← every typed arc, one query")

In [ ]:
G = nx.Graph()          # undirected, weight = number of parallel arcs
D = nx.DiGraph()        # directed (arcs have direction: who controls whom)
for s_, t_ in zip(edges["src"], edges["tgt"]):
    G.add_edge(s_, t_, weight=G[s_][t_]["weight"] + 1 if G.has_edge(s_, t_) else 1)
    D.add_edge(s_, t_)
print(f"projected graph: {G.number_of_nodes():,} nodes · {G.number_of_edges():,} edges")

def top(scores, n=10):
    """GDS 'stream' mode: top-n of a score dict, with human names."""
    return pd.DataFrame(
        [(label.get(k, k), round(v, 5)) for k, v in
         sorted(scores.items(), key=lambda kv: -kv[1])[:n]],
        columns=["actor", "score"])

## 2 · Centrality

**Degree centrality never needs an export** — it is a SPARQL `GROUP BY`,
computed inside the engine over the reified arcs:

In [ ]:
deg = g.query_df(PREFIXES + """
SELECT ?actor (COUNT(?arc) AS ?degree) WHERE {
  { ?arc lomb:source ?actor } UNION { ?arc lomb:target ?actor }
}
GROUP BY ?actor ORDER BY DESC(?degree) LIMIT 10""")
deg["actor"] = deg["actor"].map(label)
deg

For the rest of the centrality family we use the projection. Betweenness is
sampled (`k=400` pivots) exactly as GDS samples it — this is the slowest cell,
give it ~a minute. Lombardi's drawings reward it: betweenness finds the
*brokers* his drawings are literally about — BCCI, the Zurich shell
Kapital Beratung, the Harvard endowment:

In [ ]:
pagerank    = nx.pagerank(D, alpha=0.85)                      # GDS PageRank
betweenness = nx.betweenness_centrality(G, k=400, seed=1)     # GDS Betweenness (sampled)
closeness   = nx.closeness_centrality(G)                      # GDS Closeness
eigenvector = nx.eigenvector_centrality(G, max_iter=500)      # GDS Eigenvector
hubs, auths = nx.hits(D)                                      # GDS HITS

summary = pd.DataFrame({
    "pagerank":    top(pagerank)["actor"].values,
    "betweenness": top(betweenness)["actor"].values,
    "closeness":   top(closeness)["actor"].values,
    "eigenvector": top(eigenvector)["actor"].values,
    "hits-hubs":   top(hubs)["actor"].values,
}, index=range(1, 11))
print(f"articulation points (single points of failure): {len(list(nx.articulation_points(G))):,}")
summary

## 3 · Community detection

Lombardi drew each scandal as one drawing. If community detection works, it
should **rediscover the scandals** from the merged 51-drawing graph — and it
does: Louvain lands at modularity **0.91** (an extremely well-clustered graph)
with ~70 communities, and the big ones read like his catalogue raisonné:

In [ ]:
wcc = list(nx.connected_components(G))                        # GDS WCC
scc = list(nx.strongly_connected_components(D))               # GDS SCC
louvain = nx.community.louvain_communities(G, seed=42)        # GDS Louvain
labelprop = list(nx.community.label_propagation_communities(G))
print(f"WCC: {len(wcc)} components (largest {max(map(len, wcc)):,}) · "
      f"SCC: {len(scc):,} · label-prop: {len(labelprop)} communities")
print(f"Louvain: {len(louvain)} communities · "
      f"modularity {nx.community.modularity(G, louvain):.3f}")

pd.DataFrame([
    {"size": len(c),
     "members": ", ".join(sorted(label.get(a, "?") for a in c)[:4]) + " …"}
    for c in sorted(louvain, key=len, reverse=True)[:6]])

In [ ]:
tri = sum(nx.triangles(G).values()) // 3                      # GDS Triangle Count
print(f"triangles: {tri:,} · average clustering coefficient: {nx.average_clustering(G):.3f}")
core = nx.core_number(G)                                      # GDS K-Core
kmax = max(core.values())
members = sorted(label.get(n, n) for n, k in core.items() if k == kmax)
print(f"max k-core: k={kmax} · {len(members)} actors in the graph's densest shell, e.g.:")
print("  " + ", ".join(members[:10]) + " …")

## 4 · Node Similarity — with ground truth in the file

GDS Node Similarity compares nodes by neighbourhood overlap (Jaccard). Here we
compare *drawings* by shared cast — and this dataset ships its own precomputed
answer (`lomb:sharesActorsWith` overlap nodes carrying `lomb:jaccard`), so for
once an algorithm demo can be **checked against the file itself**:

In [ ]:
dep = g.query_df(PREFIXES + "SELECT ?d ?actor WHERE { ?d lomb:depicts ?actor }")
tdf = g.query_df(PREFIXES + "SELECT ?d ?name WHERE { ?d a lomb:Drawing ; schema:name ?name }")
title = dict(zip(tdf["d"], tdf["name"]))
cast = {}
for d_, a_ in zip(dep["d"], dep["actor"]):
    cast.setdefault(d_, set()).add(a_)

shipped = g.query_df(PREFIXES + """
SELECT ?a ?b ?j WHERE {
  ?ov lomb:betweenDrawing ?a, ?b ; lomb:jaccard ?j . FILTER(STR(?a) < STR(?b))
}""")
rows, agree = [], 0
for a_, b_, j_ in zip(shipped["a"], shipped["b"], shipped["j"]):
    ours = len(cast[a_] & cast[b_]) / len(cast[a_] | cast[b_])
    agree += abs(ours - float(j_)) < 0.02
    rows.append({"drawing A": title.get(a_, a_)[:34], "drawing B": title.get(b_, b_)[:34],
                 "ours": round(ours, 3), "shipped": round(float(j_), 3)})
print(f"our Jaccard agrees with the file's own on {agree}/{len(shipped)} drawing pairs")
pd.DataFrame(sorted(rows, key=lambda r: -r["ours"])[:8])

## 5 · Topological link prediction — which doubles as entity resolution

GDS ships Adamic-Adar, Common Neighbors, Preferential Attachment and Resource
Allocation; NetworkX has all four as one-liners. The top predictions here split
into two delightful categories:

- **near-duplicates under different spellings** — "FNN" ↔ "Financial News
  Network", "Indian Springs St Bk" ↔ "Indian Springs State Bank". The dataset's
  own `lomb:sameNameAs` links duplicates with *identical* names (we checked:
  250 of its 306 pairs are byte-equal labels), so link prediction **extends the
  curators' entity resolution** to the spellings exact matching cannot catch;
- **plausible ties Lombardi never drew as one arc** — "Tory party" ↔ "Pergau
  dam" *is* the 1994 aid-for-arms affair.

In [ ]:
from itertools import combinations
# candidate pairs = non-adjacent nodes sharing >= 1 neighbour (as GDS restricts it)
cand = set()
for n in G.nodes:
    for u, v in combinations(G[n], 2):
        if not G.has_edge(u, v):
            cand.add((u, v) if u < v else (v, u))
print(f"{len(cand):,} candidate pairs at distance 2")

aa = sorted(nx.adamic_adar_index(G, cand), key=lambda x: -x[2])[:8]  # GDS Adamic Adar
ra = {(u, v): s_ for u, v, s_ in nx.resource_allocation_index(G, [p[:2] for p in aa])}
pa = {(u, v): s_ for u, v, s_ in nx.preferential_attachment(G, [p[:2] for p in aa])}

pd.DataFrame([{"actor A": label.get(u, u)[:30], "actor B": label.get(v, v)[:30],
               "adamic-adar": round(s_, 2),
               "common nbrs": len(list(nx.common_neighbors(G, u, v))),
               "resource-alloc": round(ra[(u, v)], 2),
               "pref-attach": pa[(u, v)]}
              for u, v, s_ in aa])

## 6 · Path finding

Lombardi's most famous drawing connects George W. Bush to the bin Laden family
through Texas oil money. Dijkstra agrees with him — and note the SPARQL
cross-check: a *property path* answers reachability inside the engine, no
export needed (it runs over all `connectedTo` arcs, chronology included, hence
the larger count):

In [ ]:
bush = next(n for n in G.nodes if "George W. Bush" in label.get(n, ""))
osama = next(n for n in G.nodes if label.get(n, "") == "Osama bin Laden")

path = nx.shortest_path(G, bush, osama)                        # GDS Dijkstra
print("  →  ".join(label.get(n, n) for n in path))

k_paths = []
import itertools
for p_ in itertools.islice(nx.shortest_simple_paths(G, bush, osama), 3):   # GDS Yen's k-shortest
    k_paths.append(" → ".join(label.get(n, n) for n in p_))
print("\nYen's 3 shortest:", *k_paths, sep="\n  ")

reach = g.query(PREFIXES +
    "SELECT (COUNT(DISTINCT ?x) AS ?n) WHERE { <%s> (lomb:connectedTo|^lomb:connectedTo)+ ?x }" % bush)
print(f"\nSPARQL property-path: {reach[0]['n'].value} actors reachable from Bush "
      f"(NetworkX component: {len(nx.node_connected_component(G, bush)):,})")

## 7 · Node embeddings — FastRP is ~15 lines of NumPy

GDS FastRP = sparse random projection + a weighted sum of powers of the
normalized adjacency matrix (Chen et al. 2019). That is three matrix products —
NumPy territory. Same CSR matrix, same iteration weights as the GDS defaults:

In [ ]:
import scipy.sparse as sp
nodes = list(G.nodes); nix = {n: i for i, n in enumerate(nodes)}
r = [nix[u] for u, v in G.edges] + [nix[v] for u, v in G.edges]
c = [nix[v] for u, v in G.edges] + [nix[u] for u, v in G.edges]
A = sp.csr_matrix((np.ones(len(r)), (r, c)), shape=(len(nodes), len(nodes)))

deg_ = np.asarray(A.sum(1)).ravel(); deg_[deg_ == 0] = 1
An = sp.diags(1 / deg_) @ A                       # row-normalized adjacency
rng = np.random.default_rng(42)
R = rng.choice([-1.0, 0.0, 1.0], size=(len(nodes), 64), p=[1/6, 2/3, 1/6]) * np.sqrt(3)
X1 = An @ R; X2 = An @ X1; X3 = An @ X2           # 1-, 2-, 3-hop views
emb = X1 + 0.8 * X2 + 0.6 * X3                    # GDS iterationWeights
emb /= np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12
print("FastRP embeddings:", emb.shape)

# GDS K-Nearest-Neighbors on the embeddings:
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=5, metric="cosine").fit(emb)
_, ind = nn.kneighbors(emb[[nix[bush]]])
print("nearest to George W. Bush in embedding space:",
      ", ".join(label.get(nodes[j], "?") for j in ind[0][1:]))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

xy = PCA(n_components=2).fit_transform(emb)
big6 = sorted(louvain, key=len, reverse=True)[:6]
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.scatter(xy[:, 0], xy[:, 1], s=3, c="#cccccc", label="_")
for i, comm in enumerate(big6):
    ix = [nix[n] for n in comm if n in nix]
    ax.scatter(xy[ix, 0], xy[ix, 1], s=6, label=f"community {i} · {len(comm)} actors")
ax.legend(fontsize=8, loc="best"); ax.set_axis_off()
ax.set_title("FastRP embedding (PCA) coloured by Louvain community —\neach cluster ≈ one Lombardi scandal")
plt.show()

## 8 · The ML pipeline — GDS node classification, honestly

GDS node-classification pipelines = features + train/test split + a classifier.
Every actor here is typed `lo:Person` or `lo:Institution` — can structure alone
recover that? **Almost not**: FastRP-only scores ≈ the majority baseline.

The fix is the RDF advantage. In a property graph the arc *types* live in the
projection config; in RDF they are first-class data. Count each node's incident
arcs **by type** and the classifier works — and is interpretable: institutions
*receive* `InfluenceControl` arcs, people *exert* them. Lombardi would nod.

In [ ]:
types_df = g.query_df(PREFIXES + """
SELECT ?actor ?cls WHERE {
  ?actor a ?cls ; lomb:actorId ?id . FILTER(?cls IN (lo:Person, lo:Institution))
}""")
cls_map = dict(zip(types_df["actor"], types_df["cls"]))

arc_types = sorted(edges["type"].unique()); tix = {t: i for i, t in enumerate(arc_types)}
F = np.zeros((len(nodes), 2 * len(arc_types)))
for s_, t_, ty in zip(edges["src"], edges["tgt"], edges["type"]):
    F[nix[s_], tix[ty]] += 1                       # arcs OUT, by type
    F[nix[t_], len(arc_types) + tix[ty]] += 1      # arcs IN,  by type

keep = [i for i, n in enumerate(nodes) if cls_map.get(n, "").endswith(("#Person", "#Institution"))]
y = np.array([1 if cls_map[nodes[i]].endswith("#Person") else 0 for i in keep])

from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
base = max(y.mean(), 1 - y.mean())
acc_emb  = cross_val_score(LogisticRegression(max_iter=3000), emb[keep], y, cv=5).mean()
Xs = StandardScaler().fit_transform(F[keep])
acc_feat = cross_val_score(LogisticRegression(max_iter=3000), Xs, y, cv=5).mean()
print(f"majority baseline          {base:.3f}")
print(f"FastRP embeddings only     {acc_emb:.3f}   ← structure alone barely helps")
print(f"typed-edge features        {acc_feat:.3f}   ← RDF predicates are the signal")

lr = LogisticRegression(max_iter=3000).fit(Xs, y)
feat_names = [f"out:{t.split('#')[-1]}" for t in arc_types] + [f"in:{t.split('#')[-1]}" for t in arc_types]
pd.DataFrame(sorted(zip(feat_names, lr.coef_[0]), key=lambda kv: -abs(kv[1]))[:6],
             columns=["feature", "coef (+person / −institution)"]).round(2)

## 9 · Write-back — `gds.graph.writeNodeProperties`, immutably

GDS mutates the database in place. A `.rete` file is immutable — so write-back
means **deriving a new file**: bake the computed scores into triples, build, and
the analytics become SPARQL-queryable data (card and provenance included). This
is the same builder that made the original file:

In [ ]:
XSD = "http://www.w3.org/2001/XMLSchema#"
comm_of = {n: i for i, c in enumerate(sorted(louvain, key=len, reverse=True)) for n in c}
nt = []
for n in nodes:
    nt.append(f'<{n}> <urn:gds:pagerank> "{pagerank.get(n, 0):.6f}"^^<{XSD}double> .')
    nt.append(f'<{n}> <urn:gds:community> "{comm_of.get(n, -1)}"^^<{XSD}integer> .')

derived = (rete.Builder()
    .add("\n".join(nt))
    .card(title="Lombardi GDS scores — derived in a browser tab",
          license="CC BY-NC-SA 4.0 (derived from lombardinetworks.net)")
    .example("SELECT ?actor ?pr WHERE { ?actor <urn:gds:pagerank> ?pr } ORDER BY DESC(?pr) LIMIT 5",
             title="Top actors by PageRank")
    .graph())
print(f"derived graph: {derived.quads:,} triples — analytics as queryable RDF")

df = derived.query_df("""
SELECT ?community (COUNT(?actor) AS ?actors) WHERE {
  ?actor <urn:gds:community> ?community
} GROUP BY ?community ORDER BY DESC(?actors) LIMIT 5""")
df

## 10 · Same matrix, bigger engines — scikit-network

The CSR matrix `A` we built for FastRP is *exactly* what
[scikit-network](https://scikit-network.readthedocs.io) consumes — its Louvain
and PageRank are compiled and orders of magnitude faster than NetworkX, which is
what you want past ~10⁶ edges. Its wheels aren't in Pyodide, so this cell runs
when you execute the notebook in native Python (`pip install scikit-network`):

In [ ]:
try:
    from sknetwork.ranking import PageRank
    from sknetwork.clustering import Louvain
    pr = PageRank().fit_predict(A)
    communities = Louvain().fit_predict(A)
    print("scikit-network PageRank top-5:",
          ", ".join(label.get(nodes[i], "?") for i in np.argsort(-pr)[:5]))
    print(f"scikit-network Louvain: {len(set(communities))} communities")
except ImportError:
    print("scikit-network is not available in Pyodide — run this notebook in native")
    print("Python (pip install rete-graph scikit-network) and this cell lights up;")
    print("A (csr_matrix) and the edge list are already in the right shape.")

## 11 · The GNN bridge — PyTorch Geometric

GDS's GraphSAGE/HashGNN and its ML pipelines are, in the open ecosystem,
[PyTorch Geometric](https://pytorch-geometric.readthedocs.io). The projection
DataFrame maps 1:1 onto PyG's two graph containers:

- **`Data`** — `edge_index` is just our `src/tgt` columns factorized;
- **`HeteroData`** — one relation per `arcType`, which is *exactly the RDF shape*:
  `("actor", "FinancialTransaction", "actor")`.

Verified natively on this projection: a 2-layer GCN reaches **0.695** test
accuracy on Person-vs-Institution (baseline 0.576) — and, a lesson worth keeping,
the humble typed-edge logistic regression above (0.81) beats it at this scale.
Torch isn't in Pyodide, so like the previous cell this one activates natively
(`pip install torch torch_geometric`):

In [ ]:
try:
    import torch
    from torch_geometric.data import Data, HeteroData
    from torch_geometric.nn import GCNConv
    import torch.nn.functional as Fn

    ei = torch.tensor([[nix[s_] for s_ in edges["src"]],
                       [nix[t_] for t_ in edges["tgt"]]], dtype=torch.long)
    ei = torch.cat([ei, ei.flip(0)], dim=1)
    X = torch.tensor(F, dtype=torch.float)                 # typed-edge features
    y_all = torch.full((len(nodes),), -1, dtype=torch.long)
    for i in keep:
        y_all[i] = 1 if cls_map[nodes[i]].endswith("#Person") else 0
    data = Data(x=X, edge_index=ei, y=y_all)
    print(data)

    hetero = HeteroData(); hetero["actor"].num_nodes = len(nodes)
    for t_ in arc_types:
        m = edges["type"] == t_
        hetero["actor", t_.split("#")[-1], "actor"].edge_index = torch.tensor(
            [[nix[s_] for s_ in edges["src"][m]], [nix[x] for x in edges["tgt"][m]]])
    print("hetero relations:", [r for _, r, _ in hetero.edge_types])

    class GCN(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.c1, self.c2 = GCNConv(X.shape[1], 32), GCNConv(32, 2)
        def forward(self, d):
            return self.c2(Fn.relu(self.c1(d.x, d.edge_index)), d.edge_index)

    torch.manual_seed(0)
    lab = y_all >= 0
    perm = torch.randperm(int(lab.sum()))
    li = lab.nonzero(as_tuple=True)[0][perm]
    tr, te = li[:int(0.7 * len(li))], li[int(0.7 * len(li)):]
    model, opt = GCN(), None
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(120):
        model.train(); opt.zero_grad()
        Fn.cross_entropy(model(data)[tr], y_all[tr]).backward(); opt.step()
    model.eval()
    acc = (model(data).argmax(1)[te] == y_all[te]).float().mean().item()
    print(f"GCN test accuracy: {acc:.3f}")
except ImportError:
    print("PyTorch (Geometric) is not available in Pyodide — natively:")
    print("  pip install rete-graph torch torch_geometric")
    print("then this cell builds Data, HeteroData and trains the GCN as-is.")

---

## What stays native, and where this scales

- **Leiden, Node2Vec, GraphSAGE, HashGNN, CELF** — native Python
  (`sknetwork`/`leidenalg`, `gensim`, PyG), same projection code.
- **Billions of edges** — don't stream them into a browser. The same `.rete`
  files carry Parquet companions for bulk analytics, and any file is a standard
  SPARQL 1.1 endpoint, so server-side tooling (including Neo4j itself, via
  neosemantics) can consume it.
- **Write-back** here means *derive a new immutable file* — analytics results
  stay versioned, citable data instead of mutated database state.

**Where next:** [the playground](https://caviri.github.io/rete/playground.html)
(this dataset is `lombardi`) · [Python API](https://caviri.github.io/rete/python.html) ·
[agent frameworks over .rete](https://caviri.github.io/rete/agent-frameworks.html) ·
[docs index](https://caviri.github.io/rete/)

Source & issues: **<https://github.com/caviri/rete>**

© 2026 Carlos Vivar Ríos — Apache License 2.0. Dataset: *Mark Lombardi
Networks* ([lombardinetworks.net](https://lombardinetworks.net)),
CC BY-NC-SA 4.0; drawings © the estate of Mark Lombardi.